# Simple, fast baseline: Logistic Regression on one-hot / scaled features.

Purpose: establish the "floor" score before trying gradient boosting.
Everything downstream (03/04/05) should beat this comfortably.


In [2]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Paths / constants

In [3]:
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

sys.path.insert(0, BASE_DIR)

TARGET = "addicted_label"
ID_COL = "id"
TRAIN_PATH = "./dataset/train_fe.csv"
TEST_PATH = "./dataset/test_fe.csv"

MODEL_NAME = "baseline_logreg"
ARTIFACT_DIR = "./artifacts"
MODEL_DIR = "./models"
SUB_PATH = f"{ARTIFACT_DIR}/test_pred_{MODEL_NAME}.csv"
OOF_PATH = f"{ARTIFACT_DIR}/oof_{MODEL_NAME}.npy"

FOLD_IDS_PATH = f"{ARTIFACT_DIR}/fold_ids.npy"
N_SPLITS = 5
SEED = 42

NUM_COLS = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

In [4]:
YES_NO_MAP = {"Yes": 1, "No": 0}
STRESS_MAP = {"Low": 0, "Medium": 1, "High": 2}

def load_raw_data(train_path=TRAIN_PATH, test_path=TEST_PATH):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test
 
 
def engineer_features(df):
    df = df.copy()
    df["missing_count"] = df[NUM_COLS + CAT_COLS].isnull().sum(axis=1)
    df["has_any_missing"] = (df["missing_count"] > 0).astype(int)
    df["stress_ord"] = df["stress_level"].map(STRESS_MAP)
    df["academic_impact_bin"] = df["academic_work_impact"].map(YES_NO_MAP)
    df["screen_to_sleep_ratio"] = df["daily_screen_time_hours"] / (df["sleep_hours"] + 1)
    df["social_to_gaming_ratio"] = df["social_media_hours"] / (df["gaming_hours"] + 1)
    df["notifications_per_app_open"] = df["notifications_per_day"] / (df["app_opens_per_day"] + 1)
    df["weekend_vs_weekday_diff"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    df["leisure_screen_hours"] = df["daily_screen_time_hours"] - df["work_study_hours"]
    df["sleep_deficit"] = (8 - df["sleep_hours"]).clip(lower=0)
    df["screen_per_notification"] = df["daily_screen_time_hours"] / (df["notifications_per_day"] + 1)
    df["high_stress_flag"] = (df["stress_level"] == "High").astype(float)
    df["low_sleep_flag"] = (df["sleep_hours"] < 6).astype(float)
    df["heavy_screen_flag"] = (
        df["daily_screen_time_hours"] > df["daily_screen_time_hours"].median()
    ).astype(float)
    return df
 
 
def get_feature_lists(df):
    engineered_num = [
        "missing_count", "has_any_missing", "stress_ord", "academic_impact_bin",
        "screen_to_sleep_ratio", "social_to_gaming_ratio", "notifications_per_app_open",
        "weekend_vs_weekday_diff", "leisure_screen_hours", "sleep_deficit",
        "screen_per_notification", "high_stress_flag", "low_sleep_flag", "heavy_screen_flag",
    ]
    num_cols = NUM_COLS + [c for c in engineered_num if c in df.columns]
    cat_cols = [c for c in CAT_COLS if c in df.columns]
    return num_cols, cat_cols

In [5]:
def make_folds(y, n_splits=N_SPLITS, seed=SEED):
    y = np.asarray(y)
    fold_ids = np.full(len(y), -1, dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (_, valid_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
        fold_ids[valid_idx] = fold
    return fold_ids
 
 
def get_or_create_folds(train_df, target_col=TARGET, n_splits=N_SPLITS,
                         seed=SEED, path=FOLD_IDS_PATH):
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    if os.path.exists(path):
        fold_ids = np.load(path)
        if len(fold_ids) == len(train_df):
            return fold_ids
        print("Existing fold file has wrong length, regenerating folds.")
    fold_ids = make_folds(train_df[target_col].values, n_splits=n_splits, seed=seed)
    np.save(path, fold_ids)
    return fold_ids
 
 
def fold_split(train_df, fold_ids, fold):
    train_idx = np.where(fold_ids != fold)[0]
    valid_idx = np.where(fold_ids == fold)[0]
    return train_idx, valid_idx
 
 
def summarize_oof(y_true, oof_pred, model_name="model"):
    auc = roc_auc_score(y_true, oof_pred)
    print(f"[{model_name}] OOF ROC-AUC: {auc:.5f}")
    return auc

In [6]:
from sklearn.impute import SimpleImputer

def build_pipeline(num_cols, cat_cols):
    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ]
    )
    model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
    return Pipeline([("prep", preprocessor), ("clf", model)])

In [7]:
def main():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
 
    train, test = load_raw_data()
    train = engineer_features(train)
    test = engineer_features(test)
    num_cols, cat_cols = get_feature_lists(train)
 
    y = train[TARGET].values
    fold_ids = get_or_create_folds(train, target_col=TARGET)
 
    oof_pred = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    fold_scores = []
 
    print("=" * 70)
    print(f"BASELINE: Logistic Regression ({N_SPLITS}-fold CV)")
    print("=" * 70)
 
    for fold in range(N_SPLITS):
        train_idx, valid_idx = fold_split(train, fold_ids, fold)
 
        X_train = train.loc[train_idx, num_cols + cat_cols]
        X_valid = train.loc[valid_idx, num_cols + cat_cols]
        y_train, y_valid = y[train_idx], y[valid_idx]
 
        pipe = build_pipeline(num_cols, cat_cols)
        pipe.fit(X_train, y_train)
 
        valid_pred = pipe.predict_proba(X_valid)[:, 1]
        oof_pred[valid_idx] = valid_pred
 
        fold_auc = roc_auc_score(y_valid, valid_pred)
        fold_scores.append(fold_auc)
        print(f"Fold {fold}: AUC = {fold_auc:.5f}")
 
        test_pred += pipe.predict_proba(test[num_cols + cat_cols])[:, 1] / N_SPLITS
 
    print(f"\nMean fold AUC: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
    summarize_oof(y, oof_pred, MODEL_NAME)
 
    np.save(OOF_PATH, oof_pred)
    pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred}).to_csv(SUB_PATH, index=False)
 
    print(f"\nSaved OOF predictions -> {OOF_PATH}")
    print(f"Saved test predictions -> {SUB_PATH}")

In [8]:
if __name__ == "__main__":
    main()

BASELINE: Logistic Regression (5-fold CV)
Fold 0: AUC = 0.91513
Fold 1: AUC = 0.91566
Fold 2: AUC = 0.91657
Fold 3: AUC = 0.91688
Fold 4: AUC = 0.91608

Mean fold AUC: 0.91606 (+/- 0.00063)
[baseline_logreg] OOF ROC-AUC: 0.91606

Saved OOF predictions -> ./artifacts/oof_baseline_logreg.npy
Saved test predictions -> ./artifacts/test_pred_baseline_logreg.csv
